# Мини-проект по нейронным сетям

## Вариант 1. Модель с нуля

# 1. Постановка задачи

Классификация текстовых сообщений на спам.

Данные: предобработанные email-сообщения.

Цель: обучить нейронную сеть (MLP) на TF-IDF признаках для определения спама.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, f1_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
df = pd.read_csv("Данные.csv")
df = df.dropna()

df.head()

# 2. Описание данных

In [ ]:
print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df['class'].value_counts())

# 3. Подготовка данных

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

df['body'] = df['body'].apply(clean_text)

# 4. Разделение на train / validation / test

In [ ]:
X = df['body']
y = df['class']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

# 5. Выбор модели

Для решения задачи бинарной классификации текстов была выбрана многослойная нейронная сеть (MLP, Multi-Layer Perceptron), реализованная с использованием TensorFlow/Keras.

Причины выбора:

1. После преобразования текста в TF-IDF признаки каждый документ представлен в виде числового вектора фиксированной длины.
2. MLP хорошо работает с табличными и векторными данными.
3. Архитектура относительно проста и позволяет быстро обучить модель на имеющемся датасете.
4. Использование нескольких скрытых слоев позволяет модели выявлять нелинейные зависимости между признаками.

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_val_vec = vectorizer.transform(X_val).toarray()
X_test_vec = vectorizer.transform(X_test).toarray()

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(X_train_vec.shape[1],)),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

model.summary()

# 6. Обучение модели

In [ ]:
history = model.fit(
    X_train_vec, y_train,
    validation_data=(X_val_vec, y_val),
    epochs=30,
    batch_size=32
)

# 7. Графики loss / метрик

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title("Loss Curve")
plt.show()

In [ ]:
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title("Accuracy Curve")
plt.show()

# 8. Оценка на test set

In [ ]:
pred = model.predict(X_test_vec)
pred = (pred > 0.5).astype(int)

In [ ]:
print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("F1:", f1_score(y_test, pred))
print(classification_report(y_test, pred))

In [ ]:
cm = confusion_matrix(y_test, pred)
print(cm)

В ходе работы была разработана и обучена нейронная сеть для классификации электронных сообщений на спам и не спам.

Текстовые данные были преобразованы в числовые признаки с помощью метода TF-IDF. Для решения задачи использовалась многослойная нейронная сеть (MLP), реализованная средствами TensorFlow/Keras и обученная с нуля.

После обучения модель достигла точности 98.75% на тестовой выборке. Значение F1-score составило 0.987, что свидетельствует о высоком качестве классификации и хорошем балансе между полнотой и точностью.

Полученные результаты показывают, что выбранная архитектура эффективно решает задачу фильтрации спама на данном наборе данных.